# Topic 11 — Prepare Flickr8k (8,000 Images)

Use Colab GPU for indexing and evaluation. The Streamlit web interface runs separately on your laptop. This version uses the original dataset’s **batched Parquet files** to avoid per-image HTTP 429 errors.

In [ ]:
# 01 | Install experiment dependencies (Streamlit is not needed inside Colab)
%pip -q install "transformers>=4.48,<5" "datasets>=3,<5" "huggingface_hub>=0.34,<1" "torch>=2.2" "numpy>=1.26" "pandas>=2" "Pillow>=10" "matplotlib>=3.7" "tqdm>=4.66" "accelerate>=1" "pyarrow>=17"


In [ ]:
# 02 | Verify GPU and free storage
from pathlib import Path
import shutil, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('Available disk space (GB):', round(shutil.disk_usage('/content').free / 1024**3, 1))
assert torch.cuda.is_available(), 'Switch Colab Runtime > Change runtime type > T4 GPU.'
assert shutil.disk_usage('/content').free >= 6 * 1024**3, 'At least 6 GB of free disk space is recommended.'


In [ ]:
# 03 | Upload the NEW VisioSearch source ZIP (fixed version)
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
assert len(archives) == 1, 'Upload only the NEW VisioSearch project ZIP.'
with zipfile.ZipFile(archives[0]) as archive:
    archive.extractall('/content')
print('Source extracted:', archives[0])


In [ ]:
# 04 | Enter the project directory and confirm the fixed downloader is present
%cd /content/Topic11_8K_Search_App
from pathlib import Path
import subprocess, sys, os
source = Path('scripts/build_gallery.py').read_text(encoding='utf-8')
assert 'PARQUET_REVISION' in source, 'Old project ZIP detected. Upload the UPDATED ZIP instead.'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
print('Verified: batched Parquet downloader is installed.')


In [ ]:
# 05 | Download the 8,000 images from Parquet shards and build CLIP/SigLIP image embeddings
# The process stops immediately if a download or inference operation fails.
subprocess.run([sys.executable, 'scripts/build_gallery.py',
                '--output-dir', 'data/flickr8k', '--batch-size', '16'], check=True)


In [ ]:
# 06 | Check gallery integrity before running evaluation
import json
import numpy as np
rows = [json.loads(line) for line in Path('data/flickr8k/metadata.jsonl').open(encoding='utf-8')]
counts = {s: sum(row['split']==s for row in rows) for s in ('train','dev','test')}
print('Split counts:', counts)
assert counts == {'train':6000,'dev':1000,'test':1000}, 'Unexpected dataset size or split.'
assert len(rows) == 8000
assert all((Path('data/flickr8k') / r['path']).is_file() for r in rows)
for name in ('clip', 'siglip'):
    arr = np.load(f'data/flickr8k/embeddings_{name}.npy', mmap_mode='r')
    print(name, 'embedding shape:', arr.shape)
    assert arr.ndim == 2 and arr.shape[0] == 8000
print('Gallery validation passed.')


In [ ]:
# 07 | Evaluate CLIP and SigLIP: 1,000 test images vs full 8,000-image gallery
subprocess.run([sys.executable, 'scripts/evaluate.py',
                '--index-dir', 'data/flickr8k', '--output-dir', 'results',
                '--batch-size', '64'], check=True)


In [ ]:
# 08 | Display actual test metrics (do not substitute validation results)
import pandas as pd
results_path = Path('results/benchmark_results.csv')
assert results_path.is_file(), 'Evaluation did not complete.'
results = pd.read_csv(results_path)
display(results)
assert len(results) == 4 and set(results.gallery) == {'test_only','all_8000'}


In [ ]:
# 09 | Create a SMALL results-only ZIP for review and the final report
import zipfile
results_zip = Path('/content/Topic11_Evaluation_Results.zip')
with zipfile.ZipFile(results_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for item in Path('results').rglob('*'):
        if item.is_file(): archive.write(item, item.as_posix())
    archive.write('data/flickr8k/manifest.json', 'data/flickr8k/manifest.json')
print('Results ZIP:', results_zip, '| Size (MB):', round(results_zip.stat().st_size/1024**2,2))
# For ChatGPT review, download this small archive.
from google.colab import files
files.download(str(results_zip))


In [ ]:
# 10 | Create a FULL gallery archive for the local Streamlit application
# Includes 8,000 display JPEGs and both embeddings; archive is large.
import tarfile
archive_path = Path('/content/Topic11_8K_Outputs.tar.gz')
with tarfile.open(archive_path, 'w:gz') as archive:
    archive.add('data/flickr8k', arcname='data/flickr8k')
    archive.add('results', arcname='results')
print('Full gallery archive:', archive_path, 'MB:', round(archive_path.stat().st_size/1024**2,1))


In [ ]:
# 11 | Optional: transfer the full gallery to Google Drive
EXPORT_TO_DRIVE = False
if EXPORT_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    dest = Path('/content/drive/MyDrive/Topic11_8K_Outputs.tar.gz')
    shutil.copy2(archive_path, dest)
    print('Copied to Drive:', dest)
else:
    print('Set EXPORT_TO_DRIVE=True to copy the large gallery archive to Drive.')
